In [ ]:
# Post-process DES (Modelica + URBANopt) results — shared helpers live in lib.helpers.
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from urbanopt_des.urbanopt_analysis import URBANoptAnalysis
from urbanopt_des.urbanopt_geojson import DESGeoJSON as URBANoptGeoJSON

from lib.helpers import (
    apply_default_style,
    plot_combined_grid_metric_boxplots,
    plot_combined_monthly_boxplots,
    plot_end_use_bar_all,
    plot_end_use_bar_non_connected,
    plot_grid_metric_boxplots,
    plot_load_duration_curves,
    plot_seasonal_boxplots,
    silence_common_warnings,
)

PROJECTED_CRS = "EPSG:3857"
graph_size = apply_default_style()
silence_common_warnings()

# Autoreload while we develop the projects/libraries in parallel.
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
print("loaded...")

In [ ]:
# Step 1: point at either a URBANopt project directory or a scenario directory.
# Examples:
#   ./esbe_2026/activity_07b/coincident
#   ./esbe_2026/activity_07b/coincident/run/baseline_scenario
MODEL_OUTPUT_SUBDIR = "../test_activities/esbe_2026"
urbanopt_results = (
    Path(f"{MODEL_OUTPUT_SUBDIR}/activity_07b/coincident").expanduser().resolve()
)

# Step 2: aggregated Modelica project for cross-system comparison.
modelica_agg_dir = (
    Path(f"{MODEL_OUTPUT_SUBDIR}/activity_07b/five_g_agg").expanduser().resolve()
)
if not modelica_agg_dir.exists():
    raise FileNotFoundError(
        f"Modelica aggregated path does not exist: {modelica_agg_dir}"
    )

year = 2017

# Use the URBANoptAnalysis bootstrap to resolve project paths + load URBANopt results.
uo_analysis, paths = URBANoptAnalysis.bootstrap_from_uo_results(
    urbanopt_results,
    scenario_name=None,
    year_of_data=year,
    display_name="Non-Connected",
)
uo_project_dir = paths["uo_project_dir"]
scenario_name = paths["scenario_name"]
scenario_results_dir = paths["scenario_results_dir"]
results_summary_dir = paths["results_summary_dir"]
project_geojson_filename = paths["geojson_path"]

# Cross-scenario summary directory at the project level.
project_summary_dir = uo_project_dir / "_results_summary"
project_summary_dir.mkdir(parents=True, exist_ok=True)

# Modelica result paths.
des_analysis_agg_dir = uo_project_dir
des_agg_geojson_filename = project_geojson_filename
modelica_results_dir = modelica_agg_dir / "_results_summary"
modelica_results_dir.mkdir(parents=True, exist_ok=True)

print("=== Configuration (review before continuing) ===")
print(f"URBANopt input path: {urbanopt_results}")
print(f"URBANopt project dir: {uo_project_dir}")
print(f"Scenario: {scenario_name}")
print(f"GeoJSON: {project_geojson_filename}")
print(f"Results dir: {scenario_results_dir}")
print(f"Results summary dir (scenario): {results_summary_dir}")
print(f"Results summary dir (project):  {project_summary_dir}")
print("\nModelica analysis (required):")
print(f"  Base directory: {modelica_agg_dir}")
print(f"  Results summary directory: {modelica_results_dir}")

In [ ]:
# Read Modelica result folders for the main project and for the 5G aggregated run.
modelica_results, bad_or_empty_results = (
    URBANoptAnalysis.get_list_of_valid_result_folders(des_analysis_agg_dir)
)
if bad_or_empty_results:
    print(f"Found {len(bad_or_empty_results)} bad or empty results")
    for key, value in bad_or_empty_results.items():
        print(f"  {key} with {value['error']}")

five_g_agg_modelica_results, _ = URBANoptAnalysis.get_list_of_valid_result_folders(
    modelica_agg_dir
)

# Fall back to *_res.mat scan if the structured probe came back empty —
# some agg runs write results before analysis_name.txt exists.
if not five_g_agg_modelica_results:
    fallback_results = {}
    for mat_path in sorted(modelica_agg_dir.rglob("*_res.mat")):
        result_name = mat_path.parent.name
        fallback_results[result_name] = {
            "name": result_name,
            "mat_path": mat_path,
            "error": None,
        }
    five_g_agg_modelica_results = fallback_results

if five_g_agg_modelica_results:
    print(
        f"\nFound {len(five_g_agg_modelica_results)} valid aggregated results "
        f"in {modelica_agg_dir}"
    )
    for key in five_g_agg_modelica_results:
        print(f"  {key}")
else:
    raise ValueError(
        f"No valid aggregated modelica results found in {modelica_agg_dir}"
    )

# Optional filter (set names in this list to restrict to a subset).
one_to_keep = []
if one_to_keep:
    modelica_results = {k: v for k, v in modelica_results.items() if k in one_to_keep}

# Used for single-building debugging plots later.
all_results = {**modelica_results, **five_g_agg_modelica_results}
model_of_interest = next(iter(all_results))
print(f"\nModel of interest: {model_of_interest}")

# URBANopt + DES Post-Process (Connected Scenario)

This notebook post-processes a **connected DES scenario** by combining URBANopt outputs with Modelica results (including the aggregated 5G run).

What this notebook does:
1. Configures URBANopt + Modelica paths and validates key inputs.
2. Loads Modelica result folders and registers each run in `uo_analysis`.
3. Resamples and combines Modelica + OpenStudio timeseries data.
4. Computes grid metrics, rollups, summaries, and building annual metrics.
5. Creates standard comparison plots and exports summary artifacts.

Primary output locations:
- Scenario-level outputs in `run/<scenario>/output` and `run/<scenario>/_results_summary`
- Project-level cross-analysis outputs in `<project>/_results_summary`

## Process Modelica Results

This section ingests Modelica result files and prepares unified timeseries dataframes for DES comparisons.

It will:
- add each Modelica run to the active URBANopt analysis object,
- compute DES aggregations, rollups, and building summaries,
- and save processed dataframes for reuse in later notebooks.

In [ ]:
# Attach Modelica results, naming them DES 1, DES 2, ... (and DES N (5G Agg) for the aggregated set).
i = 0
for key, value in modelica_results.items():
    i += 1
    print(f"Adding Modelica results for {value['name']} from {value['mat_path']}")
    uo_analysis.add_modelica_results(value["name"], value["mat_path"])
    uo_analysis.modelica[key].display_name = f"DES {i}"
    uo_analysis.modelica[key].save_variables()

if five_g_agg_modelica_results:
    print("\n--- Processing five_g_agg Modelica results ---")
    for key, value in five_g_agg_modelica_results.items():
        i += 1
        print(f"Adding Modelica results for {value['name']} from {value['mat_path']}")
        uo_analysis.add_modelica_results(value["name"], value["mat_path"])
        uo_analysis.modelica[key].display_name = f"DES {i} (5G Agg)"
        uo_analysis.modelica[key].save_variables()

uo_analysis.modelica.keys()

In [ ]:
# If a custom ordering is needed:
# order_of_analyses = ["Non-Connected"] + [k for k in uo_analysis.modelica if k != "Non-Connected"]
# uo_analysis.sort_modelica_results_order(order_of_analyses)


In [ ]:
# Aggregated building model — read GeoJSON, align building IDs with Modelica.
geojson_agg = URBANoptGeoJSON(des_agg_geojson_filename, skip_validation=True)
print(f"GeoJSON file: {des_agg_geojson_filename}")
print(f"Number of buildings: {len(geojson_agg.get_building_ids())}")

all_modelica_keys = list(uo_analysis.modelica.keys())
if not all_modelica_keys:
    raise ValueError("No Modelica models were added to the analysis")

geojson_building_ids = geojson_agg.get_building_ids()
reference_model = all_modelica_keys[0]
modelica_building_count = uo_analysis.modelica[reference_model].number_of_buildings()

if len(geojson_building_ids) != modelica_building_count:
    warnings.warn(
        f"GeoJSON/Modelica building count mismatch "
        f"({len(geojson_building_ids)} vs {modelica_building_count}); "
        "using available Modelica buildings only for now.",
        RuntimeWarning,
    )
building_ids_for_modelica = geojson_building_ids[:modelica_building_count]

# Extra Modelica variables to gather (loop temps/flows + per-building heat / cool).
other_vars_to_gather = [
    "borFie.Q_flow",
    "TDisWatBorLvg.T",
    "TDisWatRet.T",
    "TDisWatSup.T",
    "TDisWatRet.port_a.m_flow",
    "TDisWatSup.port_a.m_flow",
    "TimeSerLoa_all_buildings.disFloCoo.PPum",
    "TimeSerLoa_all_buildings.disFloHea.PPum",
    "bui[1].bui.QReqHotWat_flow",
    "bui[1].bui.terUniCoo.mReqChiWat_flow",
    "bui[1].bui.terUniHea.mReqHeaWat_flow",
    # model specific variables
    "heaPla13e5102e.boiHotWat.TBoiLvg[1]",
    "heaPla13e5102e.boiHotWat.TBoiLvg[2]",
    "heaPla13e5102e.THWRet.T",
    "heaPla13e5102e.THWSup.T",
    "cooPla_67e4a0e1.senMasFlo.m_flow",
]
for i in range(1, modelica_building_count + 1):
    other_vars_to_gather.append(f"bui[{i}].bui.QHea_flow")
    other_vars_to_gather.append(f"bui[{i}].bui.QCoo_flow")

uo_analysis.resample_and_convert_modelica_results(
    building_ids_for_modelica, other_vars_to_gather
)
uo_analysis.save_modelica_variables()
uo_analysis.save_urbanopt_results_in_modelica_paths()
uo_analysis.combine_modelica_and_openstudio_results()
uo_analysis.resample_actual_data()
uo_analysis.save_dataframes("min_60_with_buildings")
uo_analysis.create_modelica_aggregations()

# Carbon emissions for two model years.
uo_analysis.calculate_carbon_emissions(
    "RFCE", 2024, analysis_year=2017, emissions_type="marginal", with_td_losses=True
)
uo_analysis.calculate_carbon_emissions(
    "RFCE", 2045, analysis_year=2017, emissions_type="marginal", with_td_losses=True
)

uo_analysis.create_rollups()
uo_analysis.create_building_summaries()
uo_analysis.save_dataframes()

In [ ]:
# Grid metrics + summary + per-building rollups.
uo_analysis.calculate_all_grid_metrics()
uo_analysis.save_dataframes(["grid_metrics_daily", "grid_metrics_annual"])

uo_analysis.create_summary_results()
uo_analysis.save_dataframes(["grid_summary", "end_use_summary"])
analysis_name_mappings = uo_analysis.display_name_mappings()

buildings_df = uo_analysis.create_building_level_results()
buildings_df.to_csv(
    uo_analysis.urbanopt.scenario_output_path / "building_metrics_annual.csv",
    index=True,
)
print(
    f"Saved to {uo_analysis.urbanopt.scenario_output_path / 'building_metrics_annual.csv'}"
)
display(buildings_df)

## Metrics and Plotting

This section generates summary metrics and visualizations from the combined URBANopt + DES analysis.

It includes:
- end-use summaries (non-connected and cross-analysis comparisons),
- grid-metric summaries and annual building metrics exports,
- emissions-focused comparison plots (2024 and 2045 factors),
- seasonal/load-shape diagnostics and load duration curves.

Most figures in this section are saved to the project summary directory for reporting.

In [ ]:
# Bar plot of the Non-Connected end-use summary (writes CSV + PNG into scenario summary dir).
plot_end_use_bar_non_connected(uo_analysis, results_summary_dir)

In [ ]:
# Cross-analysis bar plot of all end uses (writes into project_summary_dir).
# Pass analysis_to_exclude=[<key>, ...] to drop specific analyses.
plot_end_use_bar_all(uo_analysis, project_summary_dir)

### Notebook-Specific Diagnostics

The next cells are custom diagnostics used for this DES workflow (emissions breakdowns and single-building requested/supplied load checks).

These are helpful for QA and interpretation, but are less standardized than the shared helper-based plots.

In [ ]:
# Carbon emissions bar charts. This plot is specific to this notebook so it stays inline.
emissions_end_use_rows = [
    "Total Natural Gas Carbon Emissions",
    "Total Electricity Carbon Emissions 2024",
    "Total Electricity Carbon Emissions 2045",
    "Total Carbon Emissions 2024",
    "Total Carbon Emissions 2045",
]

df_to_plot = uo_analysis.end_use_summary.copy()
df_to_plot = df_to_plot.drop(columns="Units")
df_to_plot = df_to_plot.T
df_to_plot = df_to_plot[emissions_end_use_rows]

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for year_str, ax in zip(["2024", "2045"], axes):
    columns = [
        "Total Natural Gas Carbon Emissions",
        f"Total Electricity Carbon Emissions {year_str}",
    ]
    df_to_plot[columns].plot.bar(stacked=True, ax=ax, title=f"Carbon Year {year_str}")
    ax.set_xlabel("")
    ax.set_ylabel("Emissions [mtCO2e]")
    ax.set_ylim(0, 10000)
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()

df_total = df_to_plot[["Total Carbon Emissions 2024", "Total Carbon Emissions 2045"]]
df_total = df_total.reset_index().melt(
    id_vars="index", var_name="Carbon Year", value_name="Emissions"
)
display(df_total)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(df_total, x="index", y="Emissions", hue="Carbon Year", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Carbon Emissions [mtCO2e/year]")
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, _loc: f"{int(x):,}"))
plt.legend(title=None)
plt.tight_layout()
plt.savefig(project_summary_dir / "bar_end_use_summary_emissions.png", dpi=300)
plt.show()

In [ ]:
# Requested vs. supplied load (single-building, debugging — specific to this notebook).
df_to_plot = uo_analysis.modelica[model_of_interest].min_60_with_buildings
df_to_plot = df_to_plot[
    ["bui[1].bui.terUniHea.mReqHeaWat_flow", "bui[1].bui.terUniCoo.mReqChiWat_flow"]
]
df_to_plot = df_to_plot / 1000  # convert to kW
display(df_to_plot)
df_to_plot.plot(figsize=(10, 5), color=["red", "blue"], alpha=0.5, linewidth=0.3)
plt.ylabel("Massflow [kg/s]")
plt.xlabel("Time")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=2)
plt.tight_layout()

In [ ]:
# Per-analysis monthly + weekday energy box plots (writes to project_summary_dir).
plot_seasonal_boxplots(uo_analysis, project_summary_dir)

In [ ]:
# Combined monthly box plots — overlay every analysis.
plot_combined_monthly_boxplots(uo_analysis, project_summary_dir)

In [ ]:
# Daily grid-metric box plots per analysis.
plot_grid_metric_boxplots(uo_analysis, project_summary_dir)

In [ ]:
# Combined grid-metric box plots — overlay every analysis.
plot_combined_grid_metric_boxplots(uo_analysis, project_summary_dir)

In [ ]:
# Load duration curves — full + zoomed.
# Note: Total Natural Gas does not include the DES boiler yet.
plot_load_duration_curves(uo_analysis, project_summary_dir)

### Manually Run REopt

In [ ]:
# Export the CSV for REopt

# REopt - create the CSV data that can be uploaded to REopt
reopt_df = uo_analysis.urbanopt.data.copy()
# keep only the 'Total Energy' column
reopt_df = reopt_df[["Total Building Electricity"]] / 1000  # values need to be in kW
# rename column 'Total Energy' to 'Loads (kW)'
reopt_df = reopt_df.rename(columns={"Total Building Electricity": "Loads (kW)"})
# reset the index which is just the index, i, as it is in hours, drop the index
reopt_df["Hour"] = range(len(reopt_df))
reopt_df["Hour"] += 1
# move the column hour to be first
reopt_df = reopt_df[["Hour", "Loads (kW)"]]
# name the index Hour

display(reopt_df)

reopt_df.to_csv(
    uo_analysis.urbanopt.scenario_output_path / "REopt_loads.csv", index=False
)

# upload the CSV to REopt at https://reopt.nlr.gov/tool